# Stratified Random Sampling and Cluster Sampling

**STA 503 1.0 Sampling Techniques** · [Subject index](00_Subject_Index.ipynb)

Work through the original theory notes, calculation tables, simulations and exercises in this notebook.

## Theory contents

These notes are written for this course in our own words. Theory, worked tables, code and exercises are all in this notebook; no PDF pages are embedded.

- [Stratified design](#theory-01)
- [Weighted estimation](#theory-02)
- [Allocation rules](#theory-03)
- [Why stratification can help](#theory-04)
- [Cluster design and cost](#theory-05)
- [One-stage equal-probability cluster totals](#theory-06)
- [Two-stage sampling and interpretation](#theory-07)

## Detailed theory · Stratification, allocation and cluster sampling

<a id="theory-01"></a>

### Stratified design
Partition the population into nonoverlapping exhaustive strata before selecting samples. Sample independently within each stratum under the stated design. Useful strata group similar values or ensure subgroup representation; every stratum is sampled. Define boundaries so each unit belongs to exactly one stratum.

<a id="theory-02"></a>

### Weighted estimation
For stratum sizes $N_h$, $W_h=N_h/N$ and within-stratum SRS estimates:
$$\bar y_{st}=\sum_hW_h\bar y_h,\quad
\hat\tau_{st}=\sum_hN_h\bar y_h,$$
$$\widehat{Var}(\bar y_{st})=\sum_hW_h^2(1-n_h/N_h)s_h^2/n_h.$$
The total variance is $N^2$ times this mean variance. For a proportion, treat the within-stratum outcome as binary and use its appropriate sample variance. An unweighted average of stratum means targets a different mixture unless all stratum weights are equal.

<a id="theory-03"></a>

### Allocation rules
Equal allocation supports comparisons but may not optimize the overall mean. Proportional allocation uses $n_h=nN_h/N$. With equal per-unit costs and a fixed total sample, Neyman allocation uses $n_h\propto N_hS_h$. If per-unit costs differ, a cost-constrained optimum has $n_h\propto N_hS_h/\sqrt{c_h}$ under the usual approximation. Pilot variability estimates, integer rounding, minimum stratum sample sizes and census limits must be reconciled.

<a id="theory-04"></a>

### Why stratification can help
Removing between-stratum variability can improve precision for a fixed n when within-stratum variation is lower. It also allows separate stratum estimates and tailored fieldwork. Poorly chosen strata, extra administration or inaccurate auxiliary information can reduce the benefit.

<a id="theory-05"></a>

### Cluster design and cost
Clusters are groups such as blocks, schools or villages. Selecting clusters can reduce travel and the need for an individual-level frame. Observations within a cluster are often similar, reducing effective information relative to an SRS of the same number of individuals. A rough equal-size design effect is $1+(\bar b-1)\rho$, where $\rho$ is intracluster correlation; it is a planning approximation, not a universal variance formula.

<a id="theory-06"></a>

### One-stage equal-probability cluster totals
Select m clusters by SRS from M and observe complete totals $t_i$:
$$\hat\tau=M\bar t,\quad
\widehat{Var}(\hat\tau)=M^2(1-m/M)s_t^2/m.$$
If total element count N is known, estimate the element mean by $\hat\tau/N$. If N is unknown and cluster sizes vary, a ratio of estimated total outcome to estimated total size has different bias/variance properties and needs a ratio-estimation analysis. Do not use an unweighted mean of cluster means without checking the target.

<a id="theory-07"></a>

### Two-stage sampling and interpretation
If only some elements are observed within selected clusters, add the within-cluster stage and its weights/variance contribution. Report numbers of sampled clusters and units, selection probabilities, population totals used and design-appropriate SEs. Stratification samples from all groups; clustering samples only some groups. The distinction determines both cost and inference.

## Theory
Stratification samples within every stratum. Clustering selects only some groups and then observes their units. These designs have different sampling units and variance formulas.

For stratum $h$, let $N_h$ and $n_h$ be population and sample sizes:
$$\bar y_{st}=\sum_hW_h\bar y_h,\quad
\widehat{Var}(\bar y_{st})=\sum_hW_h^2(1-n_h/N_h)s_h^2/n_h,\quad W_h=N_h/N.$$
Do not average stratum means equally when stratum sizes differ. Proportional allocation preserves population shares; equal-cost Neyman allocation assigns more observations to large, variable strata.

For $m$ clusters selected by SRS from $M$, let $t_i$ be the total in a sampled cluster:
$$\hat\tau=M\bar t,\quad
\widehat{Var}(\hat\tau)=M^2(1-m/M)s_t^2/m.$$
The sampling fraction is at cluster level. If population element count $N$ is known, $\hat\mu=\hat\tau/N$ and its variance is the total variance divided by $N^2$.

## Worked reasoning
For two strata of sizes 200 and 800 with sample means 10 and 20, the stratified estimate is $.2(10)+.8(20)=18$. The unweighted mean 15 would target a different mixture.

## Python calculation and simulation

Predict the output, run the cell, then change one input and explain its effect.

## Worked calculation · Stratified and cluster estimates: weighted contributions

### Question and assumptions

Two strata have sizes 200 and 800, sample sizes 20 and 80, sample means 10 and 20, and sample variances 4 and 25.

### Procedure

1. Compute each population weight.
2. Calculate the weighted-mean and variance contributions by stratum.
3. Contrast this with expansion of sampled cluster totals.

### Working tables

These tables are calculated from the original teaching data below. They are visible before running Python and on GitHub.

| Stratum | N_h | n_h | W_h | Mean | W_h × mean | Variance contribution |
| --- | --- | --- | --- | --- | --- | --- |
| 1 | 200 | 20 | 0.2 | 10 | 2 | 0.0072 |
| 2 | 800 | 80 | 0.8 | 20 | 16 | 0.18 |

| Stratified result | Value |
| --- | --- |
| Mean estimate | 18 |
| Variance estimate | 0.1872 |
| SE | 0.432666 |

| Sampled cluster | Total | Deviation from mean total |
| --- | --- | --- |
| 1 | 80 | -20 |
| 2 | 100 | 0 |
| 3 | 120 | 20 |

| Cluster result | Value |
| --- | --- |
| Expanded total M × mean | 1200 |
| Total variance M²(1−m/M)s_t²/m | 14400 |

In [ ]:

from math import comb, exp, factorial, log, sqrt
from statistics import mean, median, NormalDist
from itertools import product, combinations

def average_ranks(values):
    """Assign ranks 1..n, averaging every tie group."""
    return [1 + sum(v < x for v in values) + (sum(v == x for v in values)-1)/2
            for x in values]

def show_table(headers, rows):
    def fmt(x):
        if isinstance(x, float): return f"{x:.5g}"
        return str(x).replace("|", "/")
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join("---" for _ in headers) + " |"]
    lines += ["| " + " | ".join(fmt(x) for x in row) + " |" for row in rows]
    result = "\n".join(lines)
    try:
        from IPython.display import Markdown, display
        display(Markdown(result))
    except ImportError:
        print(result)


Nh=[200,800]; nh=[20,80]; means=[10,20]; s2=[4,25]; N=sum(Nh)
rows=[]
for h,(size,n,xb,var) in enumerate(zip(Nh,nh,means,s2),1):
    w=size/N; rows.append([h,size,n,w,xb,w*xb,w*w*(1-n/size)*var/n])
show_table(["Stratum","N_h","n_h","W_h","Mean","W_h × mean","Variance contribution"],rows)
estimate=sum(r[5] for r in rows); variance=sum(r[6] for r in rows)
show_table(["Stratified result","Value"],[["Mean estimate",estimate],["Variance estimate",variance],["SE",sqrt(variance)]])
totals=[80,100,120]; M=12; m=len(totals); tb=mean(totals); st2=sum((t-tb)**2 for t in totals)/(m-1)
show_table(["Sampled cluster","Total","Deviation from mean total"],[[i,t,t-tb] for i,t in enumerate(totals,1)])
show_table(["Cluster result","Value"],[["Expanded total M × mean",M*tb],["Total variance M²(1−m/M)s_t²/m",M*M*(1-m/M)*st2/m]])
assert estimate==18


### Decision and interpretation

Strata contribute according to population size. Cluster sampling variance is calculated at the cluster-total level, rather than pretending all individuals were sampled independently.

### Your calculation check

Reproduce the table on paper, change one input in the code, and explain which intermediate columns and conclusions change. Keep the test direction, zero/tie convention and assumptions explicit.

In [ ]:
import numpy as np
rng=np.random.default_rng(503)
pop=[rng.normal(10,2,200),rng.normal(20,5,800)]
sizes=np.array([len(x) for x in pop]); W=sizes/sizes.sum(); nh=np.array([20,80])
samples=[rng.choice(x,size=int(n),replace=False) for x,n in zip(pop,nh)]
estimate=sum(w*x.mean() for w,x in zip(W,samples))
variance=sum(w*w*(1-n/N)*x.var(ddof=1)/n for w,n,N,x in zip(W,nh,sizes,samples))
print('Stratified mean:',estimate,'Estimated SE:',np.sqrt(variance))
clusters=rng.normal(100,15,size=(30,8))
chosen=rng.choice(30,size=6,replace=False); totals=clusters[chosen].sum(axis=1)
total_est=30*totals.mean(); total_var=30**2*(1-6/30)*totals.var(ddof=1)/6
print('Cluster total estimate:',total_est,'SE:',np.sqrt(total_var))
print('Known population total:',clusters.sum())


## Exercises

1. Why is the stratified estimate weighted by population sizes?
2. What is the sampling unit in the cluster calculation?
3. What happens to the finite-population variance factor if every cluster is sampled?

<details><summary>Answer sketches</summary>

1. Each stratum contributes according to its share of the target population. 2. A complete cluster, not one of its eight measurements. 3. It becomes zero because there is no sampling uncertainty about the observed finite population.
</details>